# Custom Resume Entity Extractor

**NLP Information Extraction — Extract skills, roles, education, and experience from resumes**

## Project Overview

Build a resume entity extraction pipeline using **spaCy NER** combined with
**custom EntityRuler patterns** to extract structured information: skills,
job titles, degrees, organizations, and years of experience from raw resume text.

## Learning Objectives

1. Use spaCy's EntityRuler for domain-specific entity patterns.
2. Combine rule-based and ML-based NER in one pipeline.
3. Extract structured data from semi-structured text.
4. Evaluate extraction quality with entity-level metrics.
5. Understand limitations of pattern-based IE.

## Problem Statement

Given raw resume text, extract **skills**, **job titles/roles**, **degrees**,
**organizations**, and **experience years** into a structured dictionary.

## Why This Project Matters

- Recruiters spend ~7 seconds scanning each resume — automation helps.
- ATS systems need structured data from unstructured resumes.
- Skill extraction enables job-candidate matching at scale.

## Dataset Overview

5 synthetic resume texts with known ground-truth entities.

## Dataset Source and License Notes

All resume texts are synthetic.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "spacy"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import spacy
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
nlp = spacy.load("en_core_web_sm")
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
SKILL_LIST = [
    "Python", "Java", "JavaScript", "TypeScript", "C++", "Go", "Rust", "R",
    "React", "Angular", "Vue", "Django", "Flask", "FastAPI", "Spring",
    "Docker", "Kubernetes", "AWS", "GCP", "Azure", "Terraform",
    "TensorFlow", "PyTorch", "scikit-learn", "Spark", "Hadoop",
    "SQL", "PostgreSQL", "MongoDB", "Redis", "Elasticsearch",
    "Git", "CI/CD", "Linux", "Tableau", "Power BI",
    "machine learning", "deep learning", "NLP", "computer vision",
    "data science", "data engineering",
]
DEGREE_LIST = ["BS", "MS", "PhD", "Bachelor", "Master", "MBA", "B.S.", "M.S.", "Ph.D."]
ROLE_LIST = [
    "Software Engineer", "Senior Software Engineer", "Data Scientist",
    "ML Engineer", "Machine Learning Engineer", "Data Engineer",
    "Full Stack Developer", "DevOps Engineer", "Product Manager",
    "Tech Lead", "Engineering Manager", "CTO", "VP Engineering",
]
print(f"Skills: {len(SKILL_LIST)}, Degrees: {len(DEGREE_LIST)}, Roles: {len(ROLE_LIST)}")


## Dataset Download or Loading

In [ ]:
RESUMES = [
    {"text": "John Smith — Senior Software Engineer at Google\nMS Computer Science, Stanford University, 2018\nBS Mathematics, MIT, 2016\n\nSkills: Python, Java, TensorFlow, PyTorch, Docker, AWS, SQL, Git",
     "expected_skills": ["Python","Java","TensorFlow","PyTorch","Docker","AWS","SQL","Git"], "expected_roles": ["Senior Software Engineer"], "expected_degrees": ["MS","BS"]},
    {"text": "Sarah Chen — Data Scientist\nPhD Statistics, UC Berkeley\n5+ years in machine learning and data science.\nProficient in Python, R, SQL, PyTorch, and Spark.",
     "expected_skills": ["Python","R","SQL","PyTorch","Spark"], "expected_roles": ["Data Scientist"], "expected_degrees": ["PhD"]},
    {"text": "Alex Rivera — ML Engineer at Amazon\nMaster of Science in AI, Carnegie Mellon\nSkills: Python, C++, PyTorch, TensorFlow, Docker, Kubernetes, AWS, Git",
     "expected_skills": ["Python","C++","PyTorch","TensorFlow","Docker","Kubernetes","AWS","Git"], "expected_roles": ["ML Engineer"], "expected_degrees": ["Master"]},
    {"text": "Emily Park — Full Stack Developer\nBS Computer Science, UW\nSkills: JavaScript, TypeScript, React, Python, Django, PostgreSQL, Docker, AWS, Git, CI/CD",
     "expected_skills": ["JavaScript","TypeScript","React","Python","Django","PostgreSQL","Docker","AWS","Git","CI/CD"], "expected_roles": ["Full Stack Developer"], "expected_degrees": ["BS"]},
    {"text": "David Kim — DevOps Engineer at Netflix\nMBA and BS in Computer Engineering, Georgia Tech\nSkills: Docker, Kubernetes, Terraform, AWS, GCP, Linux, Python, Git, CI/CD",
     "expected_skills": ["Docker","Kubernetes","Terraform","AWS","GCP","Linux","Python","Git","CI/CD"], "expected_roles": ["DevOps Engineer"], "expected_degrees": ["MBA","BS"]},
]
print(f"Loaded {len(RESUMES)} synthetic resumes.")


## Data Validation Checks

In [ ]:
for i, r in enumerate(RESUMES):
    assert len(r["text"]) > 50
    assert len(r["expected_skills"]) > 0
print(f"All {len(RESUMES)} resumes validated.")


## Exploratory Data Analysis

In [ ]:
stats = [{"Resume": i+1, "Words": len(r["text"].split()), "Skills": len(r["expected_skills"])} for i, r in enumerate(RESUMES)]
stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))


## Target Analysis

The targets are entity mentions: skills, roles, degrees.

## Train/Validation/Test Split Strategy

Not applicable — rule-based extraction pipeline.

## Preprocessing Strategy

spaCy EntityRuler with custom patterns for skills, roles, degrees.

## Feature Engineering — Build Extraction Pipeline

In [ ]:
# Re-load to avoid duplicate component issues
nlp = spacy.load("en_core_web_sm")
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = (
    [{"label": "SKILL", "pattern": s} for s in SKILL_LIST]
    + [{"label": "DEGREE", "pattern": d} for d in DEGREE_LIST]
    + [{"label": "ROLE", "pattern": r} for r in ROLE_LIST]
)
ruler.add_patterns(patterns)
print(f"Pipeline: {nlp.pipe_names}, Patterns: {len(patterns)}")


In [ ]:
def extract_experience_years(text):
    return [int(m) for m in re.findall(r"(\d+)\+?\s*(?:years?|yrs?)", text, re.IGNORECASE)]

def extract_resume(text):
    doc = nlp(text)
    entities = defaultdict(set)
    for ent in doc.ents:
        entities[ent.label_].add(ent.text)
    years = extract_experience_years(text)
    if years:
        entities["EXPERIENCE_YEARS"] = set(str(y) for y in years)
    return dict(entities)
print("Extraction functions ready.")


## Baseline Model — Run Extraction

In [ ]:
all_results = []
for i, r in enumerate(RESUMES):
    extracted = extract_resume(r["text"])
    all_results.append(extracted)
    print(f"\nResume {i+1}:")
    for label in ["SKILL", "ROLE", "DEGREE", "ORG", "EXPERIENCE_YEARS"]:
        if label in extracted:
            print(f"  {label:20s}: {sorted(extracted[label])}")


## LazyPredict Benchmark

> **Not applicable.** This is an NLP IE task. LazyPredict is not used.

## FLAML AutoML Run

> **Not applicable.** FLAML is not used for NLP IE tasks.

## Additional Best-Library Workflow — Evaluation

In [ ]:
def evaluate_extraction(extracted_set, expected_list):
    expected_set = set(expected_list)
    ext = extracted_set if isinstance(extracted_set, set) else set()
    tp = len(ext & expected_set)
    fp = len(ext - expected_set)
    fn = len(expected_set - ext)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"precision": prec, "recall": rec, "f1": f1}

eval_rows = []
for i, (r, ext) in enumerate(zip(RESUMES, all_results)):
    for field, key in [("SKILL","expected_skills"),("ROLE","expected_roles"),("DEGREE","expected_degrees")]:
        m = evaluate_extraction(ext.get(field, set()), r[key])
        m["Resume"] = i+1; m["Entity"] = field
        eval_rows.append(m)
eval_df = pd.DataFrame(eval_rows)
print(eval_df[["Resume","Entity","precision","recall","f1"]].to_string(index=False))


In [ ]:
avg = eval_df.groupby("Entity")[["precision","recall","f1"]].mean()
print("\nAverage Metrics:")
print(avg.round(3).to_string())
fig, ax = plt.subplots(figsize=(8,4))
avg.plot(kind="bar", ax=ax)
ax.set_title("Extraction Quality by Entity Type"); ax.set_ylim(0,1.1)
plt.tight_layout(); plt.show()


## Top 3 Model Selection

Single extraction pipeline with per-entity-type quality metrics.

## Final Training and Evaluation of Top 3

The EntityRuler + NER pipeline is the final system.

## Error Analysis

In [ ]:
print("Missed entities (false negatives):")
for i, (r, ext) in enumerate(zip(RESUMES, all_results)):
    for field, key in [("SKILL","expected_skills"),("ROLE","expected_roles")]:
        missed = set(r[key]) - ext.get(field, set())
        if missed: print(f"  Resume {i+1} [{field}]: missed {missed}")


## Interpretation and Business Insight

1. **Skills extraction has high recall** — most listed skills are caught.
2. **Role detection works well** for standard job titles.
3. **Pattern lists need maintenance** — new skills emerge constantly.
4. **spaCy's built-in NER** catches organizations and person names.
5. **Combining rules + ML** gives best coverage.

## Limitations

- Pattern-based extraction misses synonyms not in the list.
- Does not handle multi-page resumes or PDF formatting.
- English-only patterns.
- No contextual disambiguation.

## How to Improve This Project

1. Train a custom spaCy NER model on labeled resume data.
2. Use **GLiNER** for zero-shot entity extraction.
3. Add **PDF parsing** for real resumes.
4. Use **sentence embeddings** to match skills semantically.

## Production Considerations

- Integrate with ATS systems via API.
- Handle multiple resume formats (PDF, DOCX, plain text).
- GDPR/privacy compliance for personal data.
- Regular updates to skill and role dictionaries.

## Common Mistakes

1. **Overlapping patterns** — longer patterns should match first.
2. **Hardcoded lists** that become stale.
3. **Ignoring context** — extracting "C" as a skill.
4. **Not validating** against ground truth.

## Mini Challenge / Exercises

1. Add 20 more skills and test if recall improves.
2. Parse a real PDF resume and compare results.
3. Use GLiNER for zero-shot extraction.
4. Build a resume-to-JSON converter.

## Final Summary / Key Takeaways

1. **spaCy EntityRuler + NER** is effective for structured extraction.
2. **Custom patterns** catch domain-specific entities.
3. **Evaluation against ground truth** is essential.
4. **For production, combine rules with trained models.**
5. **Resume parsing** is a gateway to ATS automation.